# Logits Extractor Module

In [ ]:
def default_params(): 
    return {
        'current_model': 'M1',
        'gpu': True,
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/data/extension/mitigation/datasets',
            'current': 'prompted', # 'base' or 'prompted',
            'content_column': 'code',
            'sampling_size': 500,
            'prompt_column': 'prompt',
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'callbacks_dir' : '/workspaces/CodeSmells/datax/code_smells/callbacks/mitigation',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            ##### BY ARCHITECTURE, SAME SIZE #####
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
            'M4' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b,
            ##### BY SIZE, SAME ARCHITECTURE #####
            'S1' : 'Qwen/Qwen2.5-Coder-0.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B,
            'S2' : 'Qwen/Qwen2.5-Coder-1.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B,
            'S3' : 'Qwen/Qwen2.5-Coder-3B', #https://huggingface.co/Qwen/Qwen2.5-Coder-3B,
            'S4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
        }
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

In [4]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [5]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}"
create_folder(log_file)
log_file += '/log.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [6]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### GPU

In [7]:
! nvidia-smi

Tue Nov 18 19:33:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off |   00000000:61:00.0 Off |                    0 |
| N/A   31C    P0             33W /  250W |      14MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [8]:
torch.__version__

'2.1.2+cu121'

In [9]:
device = torch.device("cuda:0" if torch.cuda.is_available() and params['gpu'] else "cpu")
device

device(type='cuda', index=0)

In [10]:
torch.cuda.memory_allocated()

0

## Logits Extractor
>
> Extracting Tensor Logits from a given Neural Code Model
>

#### Model Loading

In [11]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir, use_fast=True)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     if params['quantization'] == 'int4':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [12]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
model.config

LlamaConfig {
  "_name_or_path": "codellama/CodeLlama-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 16384,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.36.2",
  "use_cache": true,
  "vocab_size": 32016
}

In [14]:
model.to(device) #WARNING, Verify the device before assigning to memory

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32016, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_head):

#### Dataset

In [15]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['dataset']['current']}.json")

In [16]:
df_dataset

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,n_ast_nodes,n_identifiers,s_msg_id,s_line,s_column,s_end_line,s_end_column,s_code,category,input_lenght
0,278224,ebb5e0e3f5e0b02cad2b54144022084301588ac5,keras,keras/mixed_precision/loss_scale_optimizer_tes...,loss_scale_optimizer_test.py,testHyperParametersExposed,resolve line-too-long in mixed_precision,def testHyperParametersExposed(self):\n ...,https://github.com/keras-team/keras.git,Python,...,456,22,C0304,30,0,30,73,# LossScaleOptimizer and hyperpara...,Convention,536
1,276644,84afc5193d38057e2e2badf9c889ea87d80d8fbf,keras,keras/tests/tracking_util_with_v1_optimizers_t...,tracking_util_with_v1_optimizers_test.py,__init__,Reformatting the codebase with black.\n\nPiper...,def __init__(self):\n super().__init__(...,https://github.com/keras-team/keras.git,Python,...,53,8,C0304,7,0,7,30,# pylint: disable=not-callable,Convention,62
2,24968,1a0a75e3fa896cf3c095e4146a54acef7223657e,PaddleOCR,ppstructure/pdf2word/pdf2word.py,pdf2word.py,predictAndSave,Add pdf2word exe\n\nAdd pdf2word exe,"def predictAndSave(self, imgs, img_name):\r\n ...",https://github.com/PaddlePaddle/PaddleOCR.git,Python,...,251,34,C0304,25,0,25,0,,Convention,308
3,278875,3613c3defc39c236fb1592c4f7ba1a9cc887343a,keras,keras/mixed_precision/loss_scale_optimizer_tes...,loss_scale_optimizer_test.py,testHyperParametersExposed,Remove pylint comments.\n\nPiperOrigin-RevId: ...,def testHyperParametersExposed(self):\n ...,https://github.com/keras-team/keras.git,Python,...,455,22,C0304,30,0,30,73,# LossScaleOptimizer and hyperpara...,Convention,524
4,152057,a6adc22f0711c8ab78c6ef8fc78715f815cc750f,stable-diffusion-webui,webui.py,webui.py,js,added interrupt button\nadded save button\n--a...,def js(self):\r\n obj = {\r\n ...,https://github.com/AUTOMATIC1111/stable-diffus...,Python,...,102,13,C0304,11,0,11,30,return json.dumps(obj),Convention,101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,115229,4f2861b6ded274d4a41322c107ace8107e86ebea,mindsdb,mindsdb/interfaces/database/views.py,views.py,add,store integration in sql of view (before save it),"def add(self, name, query, integration_name, c...",https://github.com/mindsdb/mindsdb.git,Python,...,247,25,W0719,21,16,21,88,"raise Exception(f""Can't find integration with ...",Warning,264
1496,282637,fae93c67adc9015c1466712f9c8ffa35a8b70872,OpenBBTerminal,bots/economy/usbonds.py,usbonds.py,usbonds_command,Refactor Bot (#1326)\n\n* First commit\r\n\r\n...,def usbonds_command():\n \n\n # Debug us...,https://github.com/OpenBB-finance/OpenBBTermin...,Python,...,469,45,W0719,12,8,12,50,"raise Exception(""No available data found"")",Warning,545
1497,2842,ccd4b9330a186090cc87e94d2da1093d45de329f,PySyft,packages/syft/src/syft/oblv/model.py,model.py,request_publish,Changes for model,"def request_publish(self, dataset_id, sigma = ...",https://github.com/OpenMined/PySyft.git,Python,...,383,31,W0719,2,12,2,116,"raise Exception(""No Domain Clients added. Set ...",Warning,497
1498,116308,326622a6fb33664de21ee1627f5f083f11b59a9e,mindsdb,mindsdb/integrations/handlers/ludwig_handler/l...,ludwig_handler.py,_learn,fix: add hyperopt,"def _learn(self, statement):\n model_na...",https://github.com/mindsdb/mindsdb.git,Python,...,374,50,W0719,8,12,8,85,"raise Exception(""Ludwig handler does not suppo...",Warning,466


In [17]:
#df_dataset = df_dataset[df_dataset['input_lenght']>=700]
#df_dataset = df_dataset[:20]

#### Logit Inference

In [18]:
def logit_extractor(model, batch, tf_encoded_inputs, from_index=0):
    """
    Output is the class CausalLMOutputWithPast (https://huggingface.co/transformers/v4.10.1/main_classes/output.html?highlight=causallmoutputwithpast)"
    logits (torch.FloatTensor of shape (batch_size, sequence_length, config.vocab_size)) – Prediction scores of the language modeling head (scores for each vocabulary token before SoftMax).
    The expression i.type(torch.LongTensor).to(device) is for casting labels for the loss
    """
    callbacks_dir = f"{params['callbacks_dir']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"
    create_folder(callbacks_dir)
    
    for idx, n in enumerate(range(from_index, len(tf_encoded_inputs), batch)):
        torch.cuda.empty_cache()
        output = []
        for encoded_sample in tf_encoded_inputs[n:n+batch]:
            output.append( 
                model(input_ids = encoded_sample, labels = encoded_sample)
            )
        output_logits = [ o['logits'].detach().to('cpu').numpy() for o in output ]  #Logits Extraction
        output_loss = np.array([ o.loss.detach().to('cpu').numpy() for o in output ])  #Language modeling loss (for next-token prediction).

        #Saving Callbacks
        current_batch = idx + (from_index//batch)
        for jdx, o_logits in enumerate(output_logits):
            np.save(f"{callbacks_dir}/logits_tensor[{jdx+n}]_batch[{current_batch}].npy", o_logits)
        np.save(f"{callbacks_dir}/_loss_batch[{current_batch}].npy", output_loss)
        
        print(f"Batch [{current_batch}] Completed")

        #Memory Released
        for out in output:
            del out.logits
            torch.cuda.empty_cache()
            del out.loss
            torch.cuda.empty_cache()
        for out in output_logits:
            del out
            torch.cuda.empty_cache()
        for out in output_loss:
            del out
            torch.cuda.empty_cache()

In [19]:
#Casting Integers to Tensor Integers. Make sure the tesor is created in a device
#We ignored the parameter attention_mask since we are not using masking here [https://huggingface.co/transformers/v4.10.1/glossary.html#attention-mask]
tf_encoded_inputs = [tokenizer(sample, return_tensors='pt')['input_ids'].to(device) for sample in df_dataset[params['dataset']['content_column']].values]

In [20]:
## ACTUAL EXPERIMENT
## TIME AND MEMORY CONSUMING
logit_extractor(
    model = model,
    batch = 1, 
    tf_encoded_inputs = tf_encoded_inputs, 
    from_index=0
)

Batch [0] Completed
Batch [1] Completed
Batch [2] Completed
Batch [3] Completed
Batch [4] Completed
Batch [5] Completed
Batch [6] Completed
Batch [7] Completed
Batch [8] Completed
Batch [9] Completed
Batch [10] Completed
Batch [11] Completed
Batch [12] Completed
Batch [13] Completed
Batch [14] Completed
Batch [15] Completed
Batch [16] Completed
Batch [17] Completed
Batch [18] Completed
Batch [19] Completed
Batch [20] Completed
Batch [21] Completed
Batch [22] Completed
Batch [23] Completed
Batch [24] Completed
Batch [25] Completed
Batch [26] Completed
Batch [27] Completed
Batch [28] Completed
Batch [29] Completed
Batch [30] Completed
Batch [31] Completed
Batch [32] Completed
Batch [33] Completed
Batch [34] Completed
Batch [35] Completed
Batch [36] Completed
Batch [37] Completed
Batch [38] Completed
Batch [39] Completed
Batch [40] Completed
Batch [41] Completed
Batch [42] Completed
Batch [43] Completed
Batch [44] Completed
Batch [45] Completed
Batch [46] Completed
Batch [47] Completed
Ba

In [21]:
print("================================= PROCESS COMPLETE =================================")

================================= PROCESS COMPLETE =================================


In [22]:
del model
torch.cuda.empty_cache()
gc.collect()

24